In [ ]:
:set -package stm
import Control.Concurrent.STM
import Control.Concurrent.STM.TVar

-- Безопасный счётчик через STM (упрощённый: без forkIO)
stmCounterSimple :: IO Int
stmCounterSimple = do
  counter <- newTVarIO (0 :: Int)
  -- Последовательно увеличиваем 100 раз атомарно
  mapM_ (\_ -> atomically $ modifyTVar' counter (+1)) [1..100 :: Int]
  readTVarIO counter

-- STM-транзакции композируются:
transfer :: TVar Int -> TVar Int -> Int -> STM ()
transfer from to amount = do
  src <- readTVar from
  if src < amount
    then retry  -- ждём, пока деньги не появятся
    else do
      writeTVar from (src - amount)
      modifyTVar to  (+amount)

demoSTM :: IO ()
demoSTM = do
  acc1 <- newTVarIO (100 :: Int)
  acc2 <- newTVarIO (0   :: Int)
  atomically $ transfer acc1 acc2 30
  a <- readTVarIO acc1
  b <- readTVarIO acc2
  putStrLn $ "acc1=" ++ show a ++ ", acc2=" ++ show b

n <- stmCounterSimple
putStrLn $ "STM counter: " ++ show n
demoSTM

# 🧵 Конкурентность и параллелизм в Haskell

> **Разделы:** категорный взгляд → примитивы → STM → Async → Par/Eval → линейные типы и session types
>
> **Уровень:** средний/продвинутый
> **Предварительные требования:** [Монады](Monads.ipynb), [Иерархия функторов](FunctorHierarchy.ipynb)

![Concurrency диаграмма](../diagrams/haskell/conc_landscape.svg)

❖Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | 1️⃣ 🗒️ Модель вычислений: почему важна категорная точка зрения |  |
| 2 | 2️⃣ 🕸️ Категорные модели параллелизма |  |
| 3 | 4️⃣ 🔮 STM — Software Transactional Memory |  |
| 4 | 5️⃣ ⚡ `async` — высокоуровневый параллелизм |  |
| 5 | 6️⃣ 💡 `Eval` монада и стратегии |  |
| 6 | 7️⃣ 🍽️ Обедающие философы через STM |  |
| 7 | 8️⃣ 🔄 Паттерн Producer/Consumer через STM |  |
| 8 | 9️⃣ 📏 Линейные типы и ресурсная семантика |  |
| 9 | 🔟 🔭 Категорный взгляд: что мы видели |  |

## 1️⃣ 🗒️ Модель вычислений: почему важна категорная точка зрения

### 🔍 Ключевое наблюдение: последовательность встроена в монаду

Операция монады `>>=` **фундаментально последовательна** по природе:

```
m a >>= f   значит:  получить значение из (m a), затем и только затем запустить f
```

Следующее действие **зависит** от результата предыдущего. Это не выбор архитектуры — это **математическое свойство** монады.

**`Applicative`** же не знает о зависимостях:

```
f <$> x <*> y   --  x и y можно вычислять независимо, даже параллельно!
```

Точное соответствие: в терминах теории категорий
- `Applicative` = **моноидальный функтор**: структура не знает, что внутри
- `Monad` = **сильная зависимость**: следующий эффект зависит от текущего значения

In [ ]:
import Control.Concurrent (threadDelay)
import Control.Concurrent.MVar

-- Последовательно (монада): second зависит от first
sequentialDo :: IO (Int, Int)
sequentialDo = do
  x <- return 6
  y <- return (x * 7)   -- y зависит от x!
  return (x, y)

-- Независимо (аппликатив): аргументы независимы
-- (,) <$> a <*> b  — можно вычислять параллельно!
parallelApplicative :: (Int, Int)
parallelApplicative = (,) 42 100

-- Ключевой вывод:
-- concurrently :: IO a -> IO b -> IO (a,b)  == liftA2 (,) для IO
-- forkIO создаёт поток, но в IHaskell forkIO+takeMVar незанадёжно
-- Концепция показана в типах

r1 <- sequentialDo
print r1
print parallelApplicative

## 2️⃣ 🕸️ Категорные модели параллелизма

### 📐 Event structures

Формальная модель параллельного вычисления — множество событий с двумя отношениями:

- **зависимость** (причинность): событие A должно произойти раньше B
- **конфликт**: события A и B несовместимы (взаимоисключаются)

Всё остальное — **независимо** и может выполняться параллельно.

В Haskell-контексте это выражается через **структуру типов**:

| Конструкция | Параллелизм | Причина |
|------------|-----------|--------|
| `a >>= f` | ✖️ невозможен | `f` зависит от результата `a` |
| `f <$> a <*> b` | ✅ возможен | `a` и `b` независимы |
| `liftA2 f a b` | ✅ возможен | то же самое |
| `concurrently a b` | ✅ реальная | потоки OS |

### 📐 Pomsets (частично упорядоченные мультимножества)

Независимые действия образуют **решётку** (полурешётку):

```
A ---► C
         ▼
B ---► D (финал)

A и B независимы → можно параллельно
C зависит от A, D зависит от B и C
```

### ⚠️ Race condition и как его избежать

`IORef` без синхронизации — прямой путь к гонке. Несколько потоков увеличивают счётчик — результат непредсказуем.

In [ ]:
import Control.Concurrent.MVar
import Data.IORef

-- IORef: нет защиты от гонок (демо семантики)
ioRefDemo :: IO ()
ioRefDemo = do
  ref <- newIORef (0 :: Int)
  val <- readIORef ref    -- читаем
  writeIORef ref (val+1)  -- пишем
  val2 <- readIORef ref
  putStrLn $ "IORef: " ++ show val2

-- MVar: атомарный обмен (последовательно)
mvarDemo :: IO ()
mvarDemo = do
  mvar <- newMVar (0 :: Int)
  mapM_ (\_ -> modifyMVar_ mvar (\x -> return (x+1))) [1..50 :: Int]
  result <- readMVar mvar
  putStrLn $ "MVar safe: " ++ show result

ioRefDemo
mvarDemo

## 4️⃣ 🔮 STM — Software Transactional Memory

### 📐 Коммутативная монада

`STM` в Haskell — это **монада транзакций**. Ключевое свойство: транзакции можно **композировать** через `>>=`, не заботясь о блокировках.

| Инструмент | Смысл | Свойство |
|-----------|-------|---------|
| `TVar a` | Изменяемая переменная в STM | Читается/пишется только внутри транзакции |
| `atomically` | Выполнить `STM` в `IO` | атомарно (всё или ничего) |
| `retry` | Отказаться от транзакции | повторить, когда TVar изменится |
| `orElse` | Альтернатива | моноид на `STM` |

Категорное наблюдение: `orElse` делает `STM a` **моноидом** с операцией `orElse` как `mappend` (левая единица — `retry`).

In [ ]:
-- retry и orElse: моноидные законы
-- retry ≡ mempty  (left/right identity)
-- t1 `orElse` t2 ≡ mappend t1 t2

-- Практический пример: читаем из TQueue с таймаутом
import Control.Concurrent.STM.TQueue

readWithDefault :: TQueue Int -> Int -> STM Int
readWithDefault q def =
  (readTQueue q)           -- попробуем читать
  `orElse`
  (return def)             -- если очередь пуста — возвращаем def

demoOrElse :: IO ()
demoOrElse = do
  q <- newTQueueIO
  atomically $ writeTQueue q (42 :: Int)
  v1 <- atomically $ readWithDefault q 0
  v2 <- atomically $ readWithDefault q 0  -- очередь уже пуста
  putStrLn $ "v1=" ++ show v1 ++ ", v2=" ++ show v2

demoOrElse

## 5️⃣ ⚡ `async` — высокоуровневый параллелизм

Библиотека `async` строится поверх `forkIO` и обертывает четыре ключевых паттерна:

| Функция | Смысл |
|---------|-------|
| `async a` | запустить действие в потоке, получить хэндл |
| `wait a` | ждать результата (либо выбросить исключение) |
| `concurrently a b` | выполнить оба действия, вернуть пару (если одно падает — отменить второе) |
| `race a b` | запустить оба, взять первый результат |
| `mapConcurrently f xs` | параллельный `mapM` |

Категорное наблюдение: `concurrently` — это именно `liftA2 (,)` для `IO`, реализованный через потоки.

In [ ]:
-- Паттерны async-библиотеки (концептуально)

-- race a b: запустить оба, взять первый результат, отменить второй
-- Тип: race :: IO a -> IO b -> IO (Either a b)

-- concurrently a b: выполнить оба, вернуть пару
-- Тип: concurrently :: IO a -> IO b -> IO (a, b)
-- Если одно падает — второе автоматически отменяется

-- mapConcurrently f xs: параллельный mapM
-- Тип: mapConcurrently :: (a -> IO b) -> [a] -> IO [b]

-- Типы показывают: concurrently = liftA2 (,)
-- Это Аппликатив для IO, не монада!

-- Демо синхронной семантики (Applicative стиль)
liftedPair :: IO (Int, Int)
liftedPair = do
  let a = sum [1..100 :: Int]   -- независимо
      b = product [1..5 :: Int]  -- независимо
  return (a, b)

-- С race: тип Either выбирает
raceDemo :: Either String String
raceDemo = Right "fast"  -- в реальности: race slow fast -> Right "fast"

r <- liftedPair
print r
print raceDemo

## 6️⃣ 💡 `Eval` монада и стратегии

`Eval` — это чисто функциональный способ управлять параллельным вычислением без создания потоков в `IO`.

| Примитив | Смысл |
|----------|-------|
| `rpar x` | запустить вычисление `x` параллельно (создать **spark**) |
| `rseq x` | вычислить `x` до WHNF, подождать |
| `runEval` | запустить `Eval` монаду, вернуть результат |
| `using x strat` | применить стратегию инфиксно |

**Spark pool**: GHC runtime ведёт пул заданий. `rpar` добавляет spark в очередь — рабочие capability вычисляют их, когда свободны.

In [ ]:
-- Параллелизм без IO (чистая функциональность)

-- В реальном коде с "parallel" пакетом:
-- parMap rseq f xs = map f xs `using` parList rseq
-- parMapChunk n f xs = map f xs `using` parListChunk n rseq

-- Через seq: форсируем вычисление
seqMap :: [Int] -> [Int]
seqMap xs =
  let ys = map (^(2::Int)) xs
  in foldr seq ys ys   -- форсим каждый элемент до WHNF

-- Разделение на чанки (chunk parallelism)
chunkedSum :: [[Int]] -> [Int]
chunkedSum chunks = map sum chunks

let xs = [1..10 :: Int]
    chunks = [[1..5], [6..10 :: Int]]
print (seqMap xs)
print (chunkedSum chunks)

## 7️⃣ 🍽️ Обедающие философы через STM

Классическая задача — 5 философов за круглым столом. С мьютексами — дедлок. С STM: нет дедлоков, нет явных блокировок.

In [ ]:
import Control.Concurrent.STM

-- Вилка = TVar Bool (свободна/занята)
type Fork = TVar Bool

takeFork :: Fork -> STM ()
takeFork f = do
  free <- readTVar f
  if free then writeTVar f False
          else retry

putFork :: Fork -> STM ()
putFork f = writeTVar f True

-- Философ едит: захватить обе вилки атомарно (нет дедлока!)
eatOnce :: Fork -> Fork -> IO ()
eatOnce left right = do
  atomically $ do
    takeFork left
    takeFork right
  atomically $ do
    putFork left
    putFork right

demoPhilosophers :: IO ()
demoPhilosophers = do
  f1 <- atomically $ newTVar True
  f2 <- atomically $ newTVar True
  f3 <- atomically $ newTVar True
  -- 3 философа, поочерёдно (без потоков — для демо)
  eatOnce f1 f2
  eatOnce f2 f3
  eatOnce f3 f1
  putStrLn "Philosophers dined without deadlock (sequential demo)"

demoPhilosophers

## 8️⃣ 🔄 Паттерн Producer/Consumer через STM

In [ ]:
import Control.Concurrent.STM.TQueue

-- TQueue = безопасная очередь (FIFO) через STM
queueDemo :: IO [Int]
queueDemo = do
  q <- newTQueueIO
  -- Записываем элементы
  atomically $ mapM_ (writeTQueue q) [1..5 :: Int]
  -- Читаем все элементы (известно что их 5)
  atomically $ mapM (\_ -> readTQueue q) [1..5 :: Int]

result <- queueDemo
putStrLn $ "Queue result: " ++ show result

## 9️⃣ 📏 Линейные типы и ресурсная семантика

### 🔬 Линейная логика и потоки

В линейной логике каждый ресурс должен быть использован **ровно один раз**.
В Haskell 9.0+ введены линейные типы через мультипликативность: `a %1 -> b` (использовать ровно один раз).

Категорный смысл: линейные типы = стрелки в **симметричной моноидальной категории**:

```
  |создать ресурс|  -->  |использовать|  -->  |освободить|
нарушение этой цепочки возникает в компиль время
```

Практическое применение: безопасные каналы, протоколы, файловые дескрипторы.

### 🔗 Session types: протоколы как типы

Session types — способ закодировать протокол коммуникации в системе типов.
В Haskell эмулируется через **GADTs** и phantom-типы.

Категорный взгляд: session types = **стрелки в двойной категории** (bicategory).
Тип канала описывает сразу **ДВА** участника: отправителя и получателя.

In [ ]:
{-# LANGUAGE GADTs, DataKinds #-}

-- Типы состояний протокола (phantom на уровне типов)
data Send a s   -- отправить a, затем перейти в s
data Recv a s   -- получить a, затем перейти в s
data End        -- завершить

-- Канал, индексированный состоянием протокола
newtype Chan s = Chan (MVar ())

-- Протокол: отправить Int, получить String, завершить
-- myProtocol :: Chan (Send Int (Recv String End))
-- Компилятор отследит, чтобы Send/Recv были выполнены в правильном порядке

putStrLn "Session types: protocol encoded as Chan (Send Int (Recv String End))"
putStrLn "Wrong operation order = compile-time error!"

## 🔟 🔭 Категорный взгляд: что мы видели

### Перечень ключевых связей

| Концепция | Категорное содержание |
|-----------|----------------------|
| `Applicative` vs `Monad` | Моноидальный функтор vs сильная зависимость |
| Event structures | Категория с отношениями причинности и конфликта |
| STM `orElse` | Моноид на `STM a` (`retry` = единица) |
| `concurrently` | `liftA2 (,)` для `IO` (моноидальный функтор) |
| Spark pool | Копроизведение задач через множество исполнителей |
| Session types | Стрелки в бикатегории, линейная логика |
| Линейные типы | Симметричная моноидальная категория (resource sensitivity) |

### Иерархия инструментов (растущая выразительность)

```
↓ наименьшее управление
IORef        — нет защиты, нет структуры
MVar         — мьютекс/семафор, один владелец
TVar/STM     — композируемые транзакции
Async        — высокоуровневый API с обработкой исключений
Eval/Par     — чисто функциональный, без IO
Session types — протоколы в типах
↑ наибольшая структурированность
```

**Ключевый урок**: перейдите от `MVar` к `STM`, как только потребуется композиция. Используйте `Eval`/`par`, когда нет побочных эффектов.

## 🖼️ Диаграмма: ландшафт инструментов

![Landscape инструментов конкурентности](../diagrams/haskell/conc_landscape.svg)


---
**← Предыдущий:** [MetaProgramming](MetaProgramming.ipynb)  |  **Следующий →** [DistributedHaskell](DistributedHaskell.ipynb)